# Agentic AI Bootcamp
## Day 4 — Memory, RAG, MCP Protocol & External Integrations

---

**Course:** 5-Day Agentic AI Bootcamp  
**Day:** 4 of 5  
**Duration:** 2 hours  
**Platform:** University Assignment Platform — Memory-Augmented & MCP-Powered Edition

---

### What you will build today

By the end of this session you will have:
- A **RAG-powered agent** that learns from every interaction using ChromaDB
- A **custom MCP server** exposing 5 tools to any agent
- An agent connected to **real external MCP servers** — GitHub, web browsing, filesystem, and more
- A **multi-MCP LangGraph pipeline** that combines persistent memory with external tools

### The 5-day arc

| Day | Theme | Key Concept | Status |
|-----|-------|-------------|--------|
| 1 | Chain-of-Thought | Prompting & reasoning | ✅ |
| 2 | ReAct + Tools | Tool use & function calling | ✅ |
| 3 | Multi-Agent LangGraph | State machines & specialisation | ✅ |
| **4** | **Memory + MCP** | **Persistent knowledge + tool standardisation** | **← Today** |
| 5 | FastAPI + WhatsApp | Production deployment | 🔜 |

## Today's Agenda

| Time | Topic |
|------|-------|
| 0:00 – 0:10 | Setup + quick review of Day 3 LangGraph |
| 0:10 – 0:30 | **Part 1:** The Memory Problem — why agents forget |
| 0:30 – 0:50 | **Part 2:** ChromaDB — your agent's long-term brain |
| 0:50 – 1:10 | **Part 3:** Full RAG agent — retrieval + generation |
| 1:10 – 1:25 | **Part 4:** MCP — the USB-C of AI tools |
| 1:25 – 1:50 | **Part 5:** External MCPs — GitHub, web, filesystem & more |
| 1:50 – 2:00 | **Part 6:** A2A preview + Lab challenge |

### What changes between Day 3 and Day 4

```
Day 3:  Question → [Planner → Researcher → Writer → Critic] → Answer
                    ↑ every run starts fresh, knows nothing from before

Day 4:  Question → Memory Retrieve → [same pipeline] → Memory Save → Answer
                    ↑ gets smarter with every question answered

        + Tools live in MCP servers (any agent can use them, not just yours)
        + External MCPs give you GitHub, web, files — for free
```

---
# Setup
Run these two cells before anything else.

In [1]:
# ── Install all required packages ─────────────────────────────────────────────
!pip install anthropic             --quiet   # Claude API client
!pip install langgraph             --quiet   # multi-agent graph framework
!pip install ddgs                  --quiet   # free web search (no API key)
!pip install requests              --quiet   # HTTP client
!pip install python-dotenv         --quiet   # load .env files
!pip install chromadb              --quiet   # vector database for persistent memory
!pip install sentence-transformers --quiet   # local embeddings (no API key needed)
!pip install mcp                   --quiet   # MCP client + server library
!pip install fastmcp               --quiet   # high-level MCP server framework
!pip install httpx                 --quiet   # async HTTP (used by mcp library)

print()
print("All packages installed successfully.")
print()
print("Package roles:")
print("  anthropic              → Claude API (powers every agent)")
print("  langgraph              → wire agents as a directed state graph")
print("  chromadb               → vector DB — the agent's long-term memory")
print("  sentence-transformers  → turn text into numbers the DB can search")
print("  mcp                    → connect to any MCP server from Python")
print("  fastmcp                → build MCP servers without boilerplate")


All packages installed successfully.

Package roles:
  anthropic              → Claude API (powers every agent)
  langgraph              → wire agents as a directed state graph
  chromadb               → vector DB — the agent's long-term memory
  sentence-transformers  → turn text into numbers the DB can search
  mcp                    → connect to any MCP server from Python
  fastmcp                → build MCP servers without boilerplate


In [2]:
import os, json, re, math, uuid, asyncio, threading, time
from pathlib import Path
from typing import TypedDict, Annotated, Literal, Optional
from dotenv import load_dotenv
import requests
import anthropic

# ── Load API key ───────────────────────────────────────────────────────────────
def find_dotenv_path():
    candidates = [
        Path.cwd() / ".env",
        Path.cwd().parent / ".env",
        Path.cwd().parent.parent / ".env",
    ]
    for c in candidates:
        if c.exists():
            return c
    return None

dotenv_path = find_dotenv_path()
if dotenv_path:
    load_dotenv(dotenv_path=dotenv_path)
    print(f"Loaded .env from: {dotenv_path}")
else:
    print("Warning: no .env found — set ANTHROPIC_API_KEY manually.")

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise OSError("ANTHROPIC_API_KEY not found. Add it to your .env file.")

client = anthropic.Anthropic()
MODEL  = "claude-haiku-4-5"
print(f"Client ready. Model: {MODEL}")

Loaded .env from: /home/administrator/Desktop/Bootcamp/.env
Client ready. Model: claude-haiku-4-5


---
# Part 1 — The Memory Problem: Why Your Agents Forget Everything

## The core issue

Every Day 3 agent starts with zero knowledge. Run it twice, it learns nothing:

```
User:  "What is Newton's second law?"
Agent: [reasons, searches, writes answer]   ← great response
       ↑ answer DISCARDED when graph ends

User:  "How does F=ma apply to falling objects?"
Agent: [searches again from scratch]        ← doesn't know it just answered Newton!
```

This is like a brilliant professor with **amnesia** — knows everything while talking to
you, remembers nothing the next day.

## Three types of memory in agentic systems

| Memory Type | Lives In | Survives restart? | Introduced |
|-------------|----------|-------------------|------------|
| **Working memory** | LangGraph state dict | ❌ Lost when graph ends | Days 1–3 |
| **Episodic memory** | ChromaDB vector store | ✅ Persists on disk | **Day 4** |
| **Semantic memory** | System prompt / hardcoded facts | ✅ But static | Days 1–4 |

## RAG = Retrieval-Augmented Generation

Instead of relying on the model's training data alone:
1. **Retrieve** relevant past answers from ChromaDB using semantic search
2. **Augment** the current prompt with that retrieved context
3. **Generate** a better answer that builds on prior knowledge

The agent gets smarter with every question it answers.

In [3]:
# ── Demo: stateless agent cannot reference a prior answer ────────────────────

def stateless_answer(question: str) -> str:
    response = client.messages.create(
        model=MODEL, max_tokens=250,
        system="You are a helpful academic assistant.",
        messages=[{"role": "user", "content": question}]
    )
    return response.content[0].text

q1 = "What is Newton's second law? Give a one-sentence definition."
q2 = "Give me a numeric example using the formula you just described."   # lies!

print("=" * 60)
print("Q1:", q1)
print("-" * 60)
a1 = stateless_answer(q1)
print(a1)

print()
print("=" * 60)
print("Q2:", q2)
print("(We're asking it to reference a formula it 'just described')")
print("-" * 60)
a2 = stateless_answer(q2)
print(a2)
print()
print("=> The agent cannot reference Q1 — it never received it.")

Q1: What is Newton's second law? Give a one-sentence definition.
------------------------------------------------------------
# Newton's Second Law

Newton's second law states that the acceleration of an object is directly proportional to the net force acting on it and inversely proportional to its mass, expressed mathematically as **F = ma** (force equals mass times acceleration).

Q2: Give me a numeric example using the formula you just described.
(We're asking it to reference a formula it 'just described')
------------------------------------------------------------
I don't have any previous context from our conversation, so I'm not sure which formula you're referring to. This appears to be the start of our chat, and I haven't described any formula yet.

Could you please:
1. **Remind me which formula** you'd like me to use, or
2. **Share the formula** you're referring to, or
3. **Describe the context** (like the topic or subject area)

Once you provide that information, I'll be happ

## The RAG Pipeline

```
┌──────────────┐   encode    ┌────────────────────┐
│  Documents   │ ──────────► │  ChromaDB Vector   │
│  (any text)  │             │      Store         │
└──────────────┘             └──────────┬─────────┘
                                        │ similarity search
                               ┌────────▼────────┐
New Question ─── encode ─────► │  Top-K Results  │
                               └────────┬────────┘
                                        │ inject as context
                               ┌────────▼────────┐
                               │  Claude Prompt  │  ← augmented!
                               └────────┬────────┘
                                        │
                               ┌────────▼────────┐
                               │  Final Answer   │
                               └─────────────────┘
```

**Key insight:** The model never needs to memorise anything.
The vector store is the memory. Claude's job is to *reason* about what's retrieved.

---
# Part 2 — ChromaDB: Your Agent's Long-Term Brain

## What is ChromaDB?

ChromaDB is an **embedded vector database** — runs inside your Python process,
no separate server, stores data locally.

A **vector** captures the *meaning* of text as an array of numbers:

```
"Newton's second law"  →  [0.12, -0.87, 0.34, ..., 0.05]   (384 numbers)
"F = ma"               →  [0.11, -0.85, 0.38, ..., 0.06]   (384 numbers)
                            ↑ similar meaning → similar numbers → close in space
"French Revolution"    →  [-0.42, 0.21, -0.67, ..., 0.88]  (very different)
```

When you ask a question, ChromaDB finds the closest vectors — **semantic search**,
not keyword matching.

## Collections = namespaced knowledge bases

```
ChromaDB instance
├── "physics_notes"    ← physics only
├── "history_notes"    ← history only
└── "assignment_qa"    ← agent memory (we build this today)
```

In [4]:
# ── Initialise ChromaDB + embedding model ─────────────────────────────────────
import chromadb
from sentence_transformers import SentenceTransformer

EMBED_MODEL = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Embedding model: all-MiniLM-L6-v2")
print(f"Output dimension: {EMBED_MODEL.get_sentence_embedding_dimension()} numbers per text")

# In-memory client (use chromadb.PersistentClient('./data/chroma') to save to disk)
chroma = chromadb.Client()

physics_col = chroma.get_or_create_collection("physics_notes")
history_col = chroma.get_or_create_collection("history_notes")

print(f"\nChromaDB ready. Collections: {[c.name for c in chroma.list_collections()]}")

/home/administrator/miniconda3/envs/bootcamp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13444.96it/s]


Embedding model: all-MiniLM-L6-v2
Output dimension: 384 numbers per text

ChromaDB ready. Collections: ['physics_notes', 'history_notes']


/tmp/ipykernel_4032125/2806141721.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Output dimension: {EMBED_MODEL.get_sentence_embedding_dimension()} numbers per text")


In [5]:
# ── Store documents, then query by semantic similarity ────────────────────────

physics_docs = [
    "Newton's First Law: An object at rest stays at rest unless acted on by an external force.",
    "Newton's Second Law: Force equals mass times acceleration. F = ma.",
    "Newton's Third Law: For every action there is an equal and opposite reaction.",
    "Kinetic energy: KE = 0.5 * m * v^2 — energy of a moving object.",
    "Gravitational potential energy: PE = mgh near Earth's surface (g = 9.81 m/s^2).",
]

history_docs = [
    "The French Revolution began in 1789 with the storming of the Bastille.",
    "The Declaration of the Rights of Man was adopted in August 1789.",
    "Napoleon Bonaparte became Emperor of France in 1804 after the Revolution.",
    "The Reign of Terror (1793-1794) saw thousands of executions under Robespierre.",
]

def store_docs(collection, docs, subject):
    embeddings = EMBED_MODEL.encode(docs).tolist()
    ids = [f"{subject}_{i}" for i in range(len(docs))]
    metadatas = [{"subject": subject, "idx": i} for i in range(len(docs))]
    collection.add(documents=docs, embeddings=embeddings, ids=ids, metadatas=metadatas)
    print(f"  Stored {len(docs)} docs in '{collection.name}'")

def retrieve(collection, query: str, n: int = 2) -> list:
    q_embed = EMBED_MODEL.encode([query]).tolist()
    results = collection.query(query_embeddings=q_embed, n_results=n)
    return results["documents"][0]

store_docs(physics_col, physics_docs, "physics")
store_docs(history_col, history_docs, "history")

print()
print("Query: 'How does force affect acceleration?'")
for i, doc in enumerate(retrieve(physics_col, "How does force affect acceleration?"), 1):
    print(f"  [{i}] {doc[:100]}")

print()
print("Query: 'violence and executions in France'")
for i, doc in enumerate(retrieve(history_col, "violence and executions in France"), 1):
    print(f"  [{i}] {doc[:100]}")

  Stored 5 docs in 'physics_notes'
  Stored 4 docs in 'history_notes'

Query: 'How does force affect acceleration?'
  [1] Newton's Second Law: Force equals mass times acceleration. F = ma.
  [2] Newton's Third Law: For every action there is an equal and opposite reaction.

Query: 'violence and executions in France'
  [1] The Reign of Terror (1793-1794) saw thousands of executions under Robespierre.
  [2] The French Revolution began in 1789 with the storming of the Bastille.


In [6]:
# ── Metadata filtering — search within a subset of documents ─────────────────

study_col = chroma.get_or_create_collection("study_notes")

notes = [
    ("Newton's laws apply to everyday objects — cars, balls, rockets.",
     {"chapter": "3", "topic": "mechanics", "difficulty": "easy"}),
    ("Quantum mechanics describes atomic-scale behaviour where Newton's laws break down.",
     {"chapter": "9", "topic": "quantum", "difficulty": "hard"}),
    ("Special relativity corrects Newton's laws at speeds near light.",
     {"chapter": "10", "topic": "relativity", "difficulty": "hard"}),
    ("Thermodynamics studies heat transfer and energy conversion in systems.",
     {"chapter": "5", "topic": "thermodynamics", "difficulty": "medium"}),
    ("The Pythagorean theorem: a² + b² = c² for right triangles.",
     {"chapter": "2", "topic": "geometry", "difficulty": "easy"}),
]

embeddings = EMBED_MODEL.encode([n[0] for n in notes]).tolist()
study_col.add(
    documents=[n[0] for n in notes],
    embeddings=embeddings,
    ids=[f"note_{i}" for i in range(len(notes))],
    metadatas=[n[1] for n in notes]
)

query = "laws governing motion of objects"
q_embed = EMBED_MODEL.encode([query]).tolist()

print(f"Query: '{query}'")
print()
print("Filter: difficulty=easy")
r = study_col.query(query_embeddings=q_embed, n_results=3, where={"difficulty": "easy"})
for doc, meta in zip(r["documents"][0], r["metadatas"][0]):
    print(f"  [Ch.{meta['chapter']} | {meta['topic']}] {doc[:90]}")

print()
print("Filter: difficulty=hard")
r = study_col.query(query_embeddings=q_embed, n_results=3, where={"difficulty": "hard"})
for doc, meta in zip(r["documents"][0], r["metadatas"][0]):
    print(f"  [Ch.{meta['chapter']} | {meta['topic']}] {doc[:90]}")

Query: 'laws governing motion of objects'

Filter: difficulty=easy
  [Ch.3 | mechanics] Newton's laws apply to everyday objects — cars, balls, rockets.
  [Ch.2 | geometry] The Pythagorean theorem: a² + b² = c² for right triangles.

Filter: difficulty=hard
  [Ch.10 | relativity] Special relativity corrects Newton's laws at speeds near light.
  [Ch.9 | quantum] Quantum mechanics describes atomic-scale behaviour where Newton's laws break down.


In [7]:
# ── Similarity scores and upsert ─────────────────────────────────────────────

query = "energy and velocity relationship"
q_embed = EMBED_MODEL.encode([query]).tolist()

results = physics_col.query(
    query_embeddings=q_embed, n_results=5,
    include=["documents", "distances"]
)

print(f"Query: '{query}'\n")
print(f"{'Distance':>10}  {'Similarity':>12}  Document")
print("-" * 70)
for doc, dist in zip(results["documents"][0], results["distances"][0]):
    sim = 1 - (dist / 2)
    bar = "█" * int(sim * 20)
    print(f"{dist:10.4f}  {bar:<20}  {doc[:55]}...")

print("\nLower distance = more similar. Similarity > 0.5 is a good match.")
print()

# Upsert — update if exists, insert if new
updated = "Newton's Second Law (updated): F = ma. Net force = mass × acceleration. Valid for non-relativistic speeds in inertial frames."
physics_col.upsert(
    ids=["physics_1"],
    documents=[updated],
    embeddings=EMBED_MODEL.encode([updated]).tolist(),
    metadatas=[{"subject": "physics", "idx": 1, "version": "2"}]
)
print(f"Upserted physics_1 (Newton Second Law updated)")
print(f"Collection now has {physics_col.count()} documents")

Query: 'energy and velocity relationship'

  Distance    Similarity  Document
----------------------------------------------------------------------
    0.7826  ████████████          Kinetic energy: KE = 0.5 * m * v^2 — energy of a moving...
    1.2690  ███████               Newton's Second Law: Force equals mass times accelerati...
    1.3339  ██████                Gravitational potential energy: PE = mgh near Earth's s...
    1.4197  █████                 Newton's First Law: An object at rest stays at rest unl...
    1.5432  ████                  Newton's Third Law: For every action there is an equal ...

Lower distance = more similar. Similarity > 0.5 is a good match.

Upserted physics_1 (Newton Second Law updated)
Collection now has 5 documents


---
# Part 3 — Full RAG Agent: Retrieval + Generation in LangGraph

## Two new nodes wrap the existing pipeline

```
               ┌──────────────────────────────────────────────────────┐
               │                 Day 4 LangGraph                      │
               │                                                       │
   Question ──►│  memory_retrieve ──► planner ──► researcher          │
               │       │                               │              │
               │  "What did                         writer            │
               │  we learn                             │              │
               │  before?"                          critic            │
               │                                       │              │
               │                               memory_save ──► END   │
               │                          "Remember this for next time"│
               └──────────────────────────────────────────────────────┘
```

Memory nodes are invisible to the user but make every future answer better.

In [8]:
# ── Memory helper functions ───────────────────────────────────────────────────

# Use a fresh in-memory client for the agent demo
agent_chroma = chromadb.Client()
AGENT_MEM    = agent_chroma.get_or_create_collection("assignment_qa")


def memory_retrieve(query: str, n: int = 3) -> list:
    if AGENT_MEM.count() == 0:
        return []
    q_embed = EMBED_MODEL.encode([query]).tolist()
    results = AGENT_MEM.query(
        query_embeddings=q_embed,
        n_results=min(n, AGENT_MEM.count()),
        include=["documents", "distances"]
    )
    hits = []
    for doc, dist in zip(results["documents"][0], results["distances"][0]):
        sim = 1 - (dist / 2)
        if sim > 0.3:
            hits.append(f"[sim={sim:.2f}] {doc[:300]}")
    return hits


def memory_store(question: str, answer: str, score: int = 0) -> str:
    doc_id = str(uuid.uuid4())
    text   = f"Q: {question}\nA: {answer}"
    embed  = EMBED_MODEL.encode([text]).tolist()
    AGENT_MEM.add(
        ids=[doc_id], documents=[text], embeddings=embed,
        metadatas=[{"score": score}]
    )
    return f"Stored (id={doc_id[:8]}). Total: {AGENT_MEM.count()}"


# Quick sanity check
print(memory_store("What is Newton's second law?",
                   "Newton's second law states F=ma. Force equals mass times acceleration.",
                   score=9))
hits = memory_retrieve("force and acceleration formula")
print(f"Retrieved {len(hits)} past answers for 'force and acceleration formula':")
for h in hits:
    print(f"  {h[:100]}")

Stored (id=f112c1ca). Total: 1
Retrieved 1 past answers for 'force and acceleration formula':
  [sim=0.58] Q: What is Newton's second law?
A: Newton's second law states F=ma. Force equals mass tim


In [9]:
# ── RAG LangGraph agent ───────────────────────────────────────────────────────
from langgraph.graph import StateGraph, START, END

def web_search(query: str, max_results: int = 3) -> str:
    try:
        from ddgs import DDGS
        results = []
        with DDGS() as ddgs:
            for r in ddgs.text(query, max_results=max_results):
                results.append(f"• {r['title']}: {r['body'][:200]}")
        return "\n".join(results) or "No results."
    except Exception as e:
        return f"Search error: {e}"


class RAGState(TypedDict):
    question:       str
    memory_context: list
    plan:           str
    research:       str
    draft:          str
    score:          int
    final_answer:   str
    revisions:      int
    log:            list


def node_memory_retrieve(state):
    hits = memory_retrieve(state["question"], n=3)
    return {**state,
            "memory_context": hits,
            "log": state.get("log", []) + [f"[MemoryRetrieve] {len(hits)} past answers found"]}


def node_planner(state):
    mem_hint = ""
    if state["memory_context"]:
        mem_hint = "\n\nRelevant past answers:\n" + "\n".join(state["memory_context"][:2])
    resp = client.messages.create(
        model=MODEL, max_tokens=400,
        system="You are an academic planner. Break the question into 3 numbered research subtasks.",
        messages=[{"role": "user", "content": f"Question: {state['question']}{mem_hint}"}]
    )
    return {**state, "plan": resp.content[0].text,
            "log": state.get("log", []) + ["[Planner] plan created"]}


def node_researcher(state):
    mem_hint = ("Context from memory:\n" + "\n".join(state["memory_context"][:2]) + "\n\n") if state["memory_context"] else ""
    search_results = web_search(f"{state['question']} {state['plan'][:80]}")
    resp = client.messages.create(
        model=MODEL, max_tokens=600,
        system=f"You are a researcher. {mem_hint}Use search results to gather key findings.",
        messages=[{"role": "user", "content": f"Question: {state['question']}\nPlan:\n{state['plan']}\nSearch:\n{search_results}"}]
    )
    return {**state, "research": resp.content[0].text,
            "log": state.get("log", []) + ["[Researcher] findings gathered"]}


def node_writer(state):
    resp = client.messages.create(
        model=MODEL, max_tokens=600,
        system="You are an academic writer. Write a clear, well-structured answer.",
        messages=[{"role": "user", "content": f"Question: {state['question']}\nResearch:\n{state['research']}"}]
    )
    return {**state, "draft": resp.content[0].text,
            "log": state.get("log", []) + ["[Writer] draft written"]}


def node_critic(state):
    resp = client.messages.create(
        model=MODEL, max_tokens=150,
        system="You are a strict academic critic. Reply only: SCORE: <1-10>\nFEEDBACK: <one sentence>",
        messages=[{"role": "user", "content": f"Question: {state['question']}\nAnswer:\n{state['draft']}"}]
    )
    match = re.search(r"SCORE:\s*(\d+)", resp.content[0].text)
    score = int(match.group(1)) if match else 5
    return {**state, "score": score,
            "log": state.get("log", []) + [f"[Critic] score={score}"]}


def node_memory_save(state):
    msg = memory_store(state["question"], state["draft"], score=state["score"])
    return {**state, "final_answer": state["draft"],
            "log": state.get("log", []) + [f"[MemorySave] {msg}"]}


def route_critic(state):
    if state["score"] >= 7 or state.get("revisions", 0) >= 2:
        return "save"
    return "revise"


g = StateGraph(RAGState)
for name, fn in [("memory_retrieve", node_memory_retrieve), ("planner", node_planner),
                 ("researcher", node_researcher), ("writer", node_writer),
                 ("critic", node_critic), ("memory_save", node_memory_save)]:
    g.add_node(name, fn)

g.add_edge(START, "memory_retrieve")
g.add_edge("memory_retrieve", "planner")
g.add_edge("planner", "researcher")
g.add_edge("researcher", "writer")
g.add_edge("writer", "critic")
g.add_edge("memory_save", END)
g.add_conditional_edges("critic", route_critic, {"save": "memory_save", "revise": "writer"})

rag_agent = g.compile()
print("RAG LangGraph agent compiled. Nodes:", list(g.nodes.keys()))

RAG LangGraph agent compiled. Nodes: ['memory_retrieve', 'planner', 'researcher', 'writer', 'critic', 'memory_save']


In [10]:
# ── Run the RAG agent — watch memory improve the second answer ────────────────

def run_rag(question: str):
    print(f"\n{'='*65}")
    print(f"  QUESTION: {question}")
    print(f"{'='*65}")
    state = dict(question=question, memory_context=[], plan="", research="",
                 draft="", score=0, final_answer="", revisions=0, log=[])
    result = rag_agent.invoke(state)
    for entry in result["log"]:
        print(f"  {entry}")
    print(f"\nQuality score: {result['score']}/10")
    print(f"Memory context used: {len(result['memory_context'])} past answers")
    print(f"\nFinal Answer:\n{'-'*50}\n{result['final_answer'][:600]}")
    return result

# First run — memory is empty
r1 = run_rag("Explain Newton's three laws of motion with real-world examples.")

# Second run — memory now contains the first answer, should improve depth
print("\n\nRunning second question (memory should help)...")
r2 = run_rag("How does Newton's second law explain why electric cars accelerate faster?")


  QUESTION: Explain Newton's three laws of motion with real-world examples.
  [MemoryRetrieve] 1 past answers found
  [Planner] plan created
  [Researcher] findings gathered
  [Writer] draft written
  [Critic] score=9
  [MemorySave] Stored (id=5d7d6b6b). Total: 2

Quality score: 9/10
Memory context used: 1 past answers

Final Answer:
--------------------------------------------------
# Newton's Three Laws of Motion: Theoretical Foundations and Practical Applications

## Introduction

Newton's three laws of motion form the cornerstone of classical mechanics, providing a mathematical framework for understanding how objects move and interact with forces. These laws, published in 1687, remain fundamental to physics education and practical engineering applications. This essay explains each law with contemporary real-world examples to illustrate their relevance.

---

## **First Law of Motion: The Law of Inertia**

### Theoretical Framework

Newton's First Law states that an obj


Running s

---
# Part 4 — MCP: The USB-C of AI Tools

## The tool duplication problem

On Days 1–3, every agent that needed `web_search` defined it locally:

```
day2_agent.py   → def web_search(query): ...   # copy 1
day3_agent.py   → def web_search(query): ...   # copy 2
rag_agent.py    → def web_search(query): ...   # copy 3
```

DuckDuckGo API changes → update 3 files. Another student needs the tool → paste + pray.

## MCP solution: tools as network services

```
Before MCP                          After MCP
──────────────────────────          ──────────────────────────────────────
Agent A  (has copies of tools)      Agent A ──┐
Agent B  (has copies of tools)      Agent B ──┼──► MCP Server
Agent C  (has copies of tools)      Agent C ──┘      ├── web_search
                                                     ├── calculator
                                                     ├── memory_store
                                                     └── read_file
```

One server, unlimited clients. Update once, everyone benefits.

## The wire protocol — JSON-RPC

```json
// List tools
{"jsonrpc":"2.0","method":"tools/list","id":1}

// Call a tool
{"jsonrpc":"2.0","method":"tools/call",
 "params":{"name":"web_search","arguments":{"query":"AI agents"}}, "id":2}
```

Works over **stdio** (subprocess pipes) or **HTTP/SSE** (network). Language-agnostic.

In [11]:
# ── Our custom MCP server — tool schemas ──────────────────────────────────────
# (Full server code is in lab/mcp_server.py — run with: python day4/lab/mcp_server.py)

OUR_MCP_TOOLS = [
    {"name": "web_search",
     "description": "Search the web via DuckDuckGo. Returns title + snippet for each result.",
     "input_schema": {"type": "object",
                      "properties": {"query": {"type": "string"},
                                     "max_results": {"type": "integer"}},
                      "required": ["query"]}},
    {"name": "calculator",
     "description": "Evaluate a math expression. Supports +, -, *, /, **, %, sqrt, pi, abs.",
     "input_schema": {"type": "object",
                      "properties": {"expression": {"type": "string",
                                                    "description": "e.g. 'sqrt(144) + 2**8'"}},
                      "required": ["expression"]}},
    {"name": "memory_store",
     "description": "Persist a Q&A pair in ChromaDB for future retrieval.",
     "input_schema": {"type": "object",
                      "properties": {"question": {"type": "string"},
                                     "answer": {"type": "string"},
                                     "subject": {"type": "string"},
                                     "score": {"type": "number"}},
                      "required": ["question", "answer"]}},
    {"name": "memory_search",
     "description": "Retrieve past Q&A pairs semantically similar to the query.",
     "input_schema": {"type": "object",
                      "properties": {"query": {"type": "string"},
                                     "n_results": {"type": "integer"}},
                      "required": ["query"]}},
    {"name": "read_file",
     "description": "Read a .txt or .pdf file from disk (first 3000 chars).",
     "input_schema": {"type": "object",
                      "properties": {"path": {"type": "string"},
                                     "max_chars": {"type": "integer"}},
                      "required": ["path"]}},
]

print("Our MCP Server — Available Tools")
print("=" * 50)
for t in OUR_MCP_TOOLS:
    params = list(t["input_schema"]["properties"].keys())
    req    = t["input_schema"].get("required", [])
    print(f"\n  {t['name']}({', '.join(params)})")
    print(f"  {t['description']}")
    print(f"  required: {req}")

Our MCP Server — Available Tools

  web_search(query, max_results)
  Search the web via DuckDuckGo. Returns title + snippet for each result.
  required: ['query']

  calculator(expression)
  Evaluate a math expression. Supports +, -, *, /, **, %, sqrt, pi, abs.
  required: ['expression']

  memory_store(question, answer, subject, score)
  Persist a Q&A pair in ChromaDB for future retrieval.
  required: ['question', 'answer']

  memory_search(query, n_results)
  Retrieve past Q&A pairs semantically similar to the query.
  required: ['query']

  read_file(path, max_chars)
  Read a .txt or .pdf file from disk (first 3000 chars).
  required: ['path']


In [12]:
# ── Connecting to an MCP server from Python ───────────────────────────────────
# The `mcp` Python library connects to any MCP server via stdio or HTTP.

import asyncio

async def demo_mcp_import():
    try:
        from mcp import ClientSession, StdioServerParameters
        from mcp.client.stdio import stdio_client
        print("mcp library imported successfully!")
        print()
        print("To connect to our custom server:")
        print("  server_params = StdioServerParameters(")
        print("      command='python', args=['day4/lab/mcp_server.py'])")
        print("  async with stdio_client(server_params) as (read, write):")
        print("      async with ClientSession(read, write) as session:")
        print("          await session.initialize()")
        print("          tools = await session.list_tools()")
        print("          result = await session.call_tool('web_search', {'query': 'AI'})")
        print()
        print("Same pattern works for ANY MCP server — GitHub, Fetch, Filesystem, etc.")
    except ImportError:
        print("mcp not installed: pip install mcp")

# Jupyter/IPython already runs an asyncio loop; asyncio.run() raises RuntimeError there.
await demo_mcp_import()

print()
print("=" * 50)
print("Claude + native MCP support (beta):")
print()
print("  response = client.beta.messages.create(")
print("      model=MODEL,")
print("      mcp_servers=[{")
print("          'type': 'url',")
print("          'url': 'http://localhost:8000/sse',")
print("          'name': 'our-tools'")
print("      }],")
print("      betas=['mcp-client-2025-04-04']")
print("  )")
print()
print("Claude auto-discovers all tools and calls them as needed!")

mcp library imported successfully!

To connect to our custom server:
  server_params = StdioServerParameters(
      command='python', args=['day4/lab/mcp_server.py'])
  async with stdio_client(server_params) as (read, write):
      async with ClientSession(read, write) as session:
          await session.initialize()
          tools = await session.list_tools()
          result = await session.call_tool('web_search', {'query': 'AI'})

Same pattern works for ANY MCP server — GitHub, Fetch, Filesystem, etc.

Claude + native MCP support (beta):

  response = client.beta.messages.create(
      model=MODEL,
      mcp_servers=[{
          'type': 'url',
          'url': 'http://localhost:8000/sse',
          'name': 'our-tools'
      }],
      betas=['mcp-client-2025-04-04']
  )

Claude auto-discovers all tools and calls them as needed!


---
# Part 5 — External MCPs: Where It Gets REALLY Fun 🚀

## The MCP ecosystem

MCP is an open standard — anyone can publish an MCP server. There are already hundreds.

### No API key needed (runs with `npx`)

| Server | What it does | Install |
|--------|-------------|---------|
| `server-fetch` | Fetch any URL, read web pages as text | `npx -y @modelcontextprotocol/server-fetch` |
| `server-filesystem` | Read/write files on your machine | `npx -y @modelcontextprotocol/server-filesystem /path` |
| `server-sequential-thinking` | Step-by-step structured reasoning | `npx -y @modelcontextprotocol/server-sequential-thinking` |
| `server-memory` | Knowledge graph memory (entities + relations) | `npx -y @modelcontextprotocol/server-memory` |
| `server-time` | Current time + timezone conversions | `npx -y @modelcontextprotocol/server-time` |

### Needs an API key

| Server | What it does | Key |
|--------|-------------|-----|
| `server-github` | Create PRs, read repos, manage issues | `GITHUB_TOKEN` |
| `server-brave-search` | Premium web search | `BRAVE_API_KEY` |
| `server-slack` | Post/read Slack messages | `SLACK_TOKEN` |
| `server-google-maps` | Geocoding, directions, places | `GOOGLE_MAPS_KEY` |

### Community favourites

| Server | What it does |
|--------|-------------|
| `mcp-server-puppeteer` | Browser automation — screenshots, clicking, scraping |
| `mcp-server-postgres` | Query a PostgreSQL database in natural language |
| `mcp-server-sqlite` | Read/write SQLite databases |
| `mcp-server-exa` | Research-grade semantic web search |
| `mcp-notion` | Read/write Notion pages and databases |

> **Philosophy:** Your agent is a conductor, not a one-man band. 
> Compose capabilities from servers others built.

In [13]:
# ── Fetch MCP — Browse real web pages ────────────────────────────────────────
# @modelcontextprotocol/server-fetch exposes: fetch(url, max_length?)
# We implement the same interface using Python requests.

def fetch_tool(url: str, max_length: int = 3000) -> dict:
    """Mimics the Fetch MCP server's 'fetch' tool."""
    try:
        headers = {"User-Agent": "Mozilla/5.0 (MCP-Fetch/1.0)"}
        resp = requests.get(url, headers=headers, timeout=10)
        content = resp.text
        content = re.sub(r'<[^>]+>', '', content)        # strip HTML tags
        content = re.sub(r'\s+', ' ', content).strip()  # normalise whitespace
        return {
            "success": True, "url": url,
            "status_code": resp.status_code,
            "content": content[:max_length],
            "content_length": len(content)
        }
    except Exception as e:
        return {"success": False, "url": url, "error": str(e)}


print("Fetch MCP — fetching a real Wikipedia page...\n")
result = fetch_tool("https://en.wikipedia.org/wiki/Model_Context_Protocol", max_length=1200)

if result["success"]:
    print(f"URL: {result['url']}")
    print(f"Status: {result['status_code']}  |  Total content: {result['content_length']:,} chars")
    print(f"\nContent preview:")
    print("-" * 50)
    print(result["content"][:600])
    print("...")
else:
    print(f"Fetch failed: {result.get('error')}")
    # Fallback demo
    result["content"] = "The Model Context Protocol (MCP) is an open protocol by Anthropic to standardise how AI applications connect to tools and data sources."
    print("Demo content:", result["content"])

Fetch MCP — fetching a real Wikipedia page...

URL: https://en.wikipedia.org/wiki/Model_Context_Protocol
Status: 200  |  Total content: 36,173 chars

Content preview:
--------------------------------------------------
Model Context Protocol - Wikipedia (function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-theme-
...


In [14]:
# ── Filesystem MCP — Read & Write files ──────────────────────────────────────
# @modelcontextprotocol/server-filesystem exposes:
#   read_file, write_file, list_directory, create_directory, search_files

from datetime import datetime

def filesystem_tool(operation: str, **kwargs) -> dict:
    """Mimics the Filesystem MCP server tools."""
    if operation == "list_directory":
        path = Path(kwargs.get("path", "."))
        if not path.exists():
            return {"error": f"Not found: {path}"}
        items = [{"name": p.name, "type": "dir" if p.is_dir() else "file",
                  "size": p.stat().st_size if p.is_file() else None}
                 for p in sorted(path.iterdir())]
        return {"path": str(path), "items": items}

    elif operation == "read_file":
        path = Path(kwargs["path"])
        if not path.exists():
            return {"error": f"Not found: {path}"}
        content = path.read_text(errors="replace")
        max_c = kwargs.get("max_chars", 3000)
        return {"path": str(path), "content": content[:max_c],
                "size": len(content), "truncated": len(content) > max_c}

    elif operation == "write_file":
        path = Path(kwargs["path"])
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(kwargs["content"])
        return {"path": str(path), "bytes_written": len(kwargs["content"])}

    elif operation == "search_files":
        root = Path(kwargs.get("path", "."))
        pattern = kwargs.get("pattern", "*")
        matches = [str(p) for p in root.rglob(pattern) if p.is_file()]
        return {"matches": matches[:20], "pattern": pattern}

    return {"error": f"Unknown operation: {operation}"}


# Demo: write a file, then read it back
print("Filesystem MCP demos\n")

print("list_directory('.')")
r = filesystem_tool("list_directory", path=".")
for item in r.get("items", [])[:8]:
    icon = "D" if item["type"] == "dir" else "F"
    size = f" ({item['size']:,}b)" if item.get("size") else ""
    print(f"  [{icon}] {item['name']}{size}")

content = f"Written by MCP filesystem tool\nTimestamp: {datetime.now()}\nAgent memory is the future!"
wr = filesystem_tool("write_file", path="/tmp/mcp_agent_test.txt", content=content)
print(f"\nwrite_file → {wr['path']} ({wr['bytes_written']} bytes)")

rr = filesystem_tool("read_file", path="/tmp/mcp_agent_test.txt")
print(f"read_file → {rr['content']}")

Filesystem MCP demos

list_directory('.')
  [F] Day4_Agentic_AI_Bootcamp.ipynb (120,970b)
  [F] README.md (2,538b)
  [D] concepts
  [D] lab

write_file → /tmp/mcp_agent_test.txt (96 bytes)
read_file → Written by MCP filesystem tool
Timestamp: 2026-05-07 05:44:58.241335
Agent memory is the future!


In [15]:
# ── Sequential Thinking MCP — Structured Reasoning Chains ────────────────────
# @modelcontextprotocol/server-sequential-thinking forces the model to think
# step-by-step in an auditable chain before committing to an answer.
# Each "thought" is a discrete reasoning step that can be revised.

def sequential_thinking_tool(thought: str, thought_number: int, total_thoughts: int,
                              next_thought_needed: bool, is_revision: bool = False) -> dict:
    return {"thought": thought, "thoughtNumber": thought_number,
            "totalThoughts": total_thoughts, "nextThoughtNeeded": next_thought_needed,
            "isRevision": is_revision}


def think_through(problem: str) -> str:
    """Use Claude + sequential-thinking pattern to reason through a hard problem."""
    print(f"Sequential Thinking: '{problem[:60]}'\n")

    # Step 1: Generate the thinking plan
    resp = client.messages.create(
        model=MODEL, max_tokens=600,
        system=(
            "You are a careful reasoner. For the problem given, produce exactly 5 "
            "sequential thoughts building toward the answer. Format each as:\n"
            "THOUGHT 1: <content>\nTHOUGHT 2: <content>\n..."
        ),
        messages=[{"role": "user", "content": f"Problem: {problem}"}]
    )

    raw = resp.content[0].text
    thoughts = re.findall(r"THOUGHT\s+\d+:\s*(.+?)(?=THOUGHT|\Z)", raw, re.DOTALL)

    chain = []
    print("Reasoning chain:")
    print("─" * 55)
    for i, t in enumerate(thoughts[:5], 1):
        t = t.strip()
        step = sequential_thinking_tool(t, i, min(5, len(thoughts)), i < len(thoughts[:5]))
        chain.append(step)
        print(f"[{i}/{min(5,len(thoughts))}] {t[:115]}...")

    # Step 2: Synthesise the final answer
    chain_text = "\n".join(f"Step {s['thoughtNumber']}: {s['thought']}" for s in chain)
    final_resp = client.messages.create(
        model=MODEL, max_tokens=350,
        system="You are given a structured reasoning chain. Write a concise, clear final answer.",
        messages=[{"role": "user", "content": f"Problem: {problem}\n\nReasoning:\n{chain_text}\n\nFinal answer:"}]
    )
    answer = final_resp.content[0].text
    print("─" * 55)
    print(f"\nFinal Answer:\n{answer}")
    return answer


think_through(
    "Should a startup with a limited budget use RAG or fine-tuning for a domain-specific AI assistant?"
)

Sequential Thinking: 'Should a startup with a limited budget use RAG or fine-tunin'

Reasoning chain:
───────────────────────────────────────────────────────
[1/5] RAG (Retrieval-Augmented Generation) involves connecting an LLM to external documents/databases to pull relevant in...
[2/5] Budget considerations strongly favor RAG for startups because fine-tuning requires significant computational resour...
[3/5] RAG offers practical advantages for resource-constrained teams: faster iteration cycles (update documents without r...
[4/5] However, RAG has limitations that matter depending on the use case—it works best when answers exist in your documen...
[5/5] The pragmatic recommendation: startups should start with RAG as their primary approach because it's more cost-effec...
───────────────────────────────────────────────────────

Final Answer:
# RAG vs Fine-tuning for Budget-Constrained Startups

**Recommendation: Start with RAG**

For a startup with limited budget, **RAG (Retrieval-Augm

"# RAG vs Fine-tuning for Budget-Constrained Startups\n\n**Recommendation: Start with RAG**\n\nFor a startup with limited budget, **RAG (Retrieval-Augmented Generation) is the better initial choice** because:\n\n## Cost Advantage\n- RAG requires minimal computational resources (mainly document preparation and a retrieval system)\n- Fine-tuning demands expensive GPU infrastructure, specialized ML expertise, and ongoing maintenance\n- RAG enables faster iteration without retraining cycles\n\n## Practical Benefits\n- Easier to debug and audit (transparent source documents)\n- Simpler to update domain knowledge (refresh documents, not retrain models)\n- Faster time-to-market\n- Lower barrier to implementation\n\n## When to Consider Fine-tuning Later\nOnly graduate to fine-tuning if:\n- RAG performance plateaus despite optimization efforts\n- Your use case requires deep pattern recognition beyond retrieval (complex reasoning, mathematical operations)\n- Business metrics justify the investme

In [16]:
# ── GitHub MCP — Your Agent Manages Code ─────────────────────────────────────
# @modelcontextprotocol/server-github (needs GITHUB_TOKEN) exposes:
#   search_repositories, get_file_contents, create_issue, list_issues,
#   create_pull_request, fork_repository, create_branch, list_commits ...
#
# We call the GitHub REST API directly — identical operations.

def github_tool(operation: str, **kwargs) -> dict:
    token = os.environ.get("GITHUB_TOKEN", "")
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28"
    }
    if token:
        headers["Authorization"] = f"Bearer {token}"
    base = "https://api.github.com"

    try:
        if operation == "search_repositories":
            q = kwargs.get("query", "mcp server")
            r = requests.get(f"{base}/search/repositories?q={q}&sort=stars&per_page=6",
                             headers=headers, timeout=10)
            items = r.json().get("items", [])
            return {"repos": [{"name": i["full_name"],
                               "stars": i["stargazers_count"],
                               "description": (i.get("description") or "")[:80]}
                              for i in items]}

        elif operation == "get_file_contents":
            owner, repo, path = kwargs["owner"], kwargs["repo"], kwargs.get("path", "README.md")
            r = requests.get(f"{base}/repos/{owner}/{repo}/contents/{path}",
                             headers=headers, timeout=10)
            if r.status_code == 200:
                import base64
                data = r.json()
                content = base64.b64decode(data["content"]).decode("utf-8", errors="replace")
                return {"path": path, "content": content[:2000], "sha": data["sha"]}
            return {"error": r.json().get("message", "Not found")}

        elif operation == "list_issues":
            owner, repo = kwargs["owner"], kwargs["repo"]
            r = requests.get(f"{base}/repos/{owner}/{repo}/issues?per_page=5&state=open",
                             headers=headers, timeout=10)
            issues = r.json() if isinstance(r.json(), list) else []
            return {"issues": [{"number": i["number"], "title": i["title"],
                                "labels": [l["name"] for l in i.get("labels", [])]}
                               for i in issues[:5]]}

    except Exception as e:
        return {"error": str(e)}

    return {"error": f"Unknown operation: {operation}"}


# Demo 1: Search GitHub for popular MCP projects
print("GitHub MCP — search_repositories()")
result = github_tool("search_repositories", query="modelcontextprotocol server stars:>50")
if "repos" in result:
    print("Top MCP-related repositories:")
    for r in result["repos"]:
        print(f"  {'⭐':} {r['stars']:>6,}  {r['name']}")
        if r['description']:
            print(f"           {r['description']}")
else:
    print(f"  {result}")

# Demo 2: Read a file from a public repo
print("\nGitHub MCP — get_file_contents()")
result = github_tool("get_file_contents",
                     owner="modelcontextprotocol", repo="servers", path="README.md")
if "content" in result:
    print(f"README.md (first 400 chars):\n{result['content'][:400]}")
else:
    print(f"  {result}")

GitHub MCP — search_repositories()
Top MCP-related repositories:
  ⭐  9,895  mcp-use/mcp-use
           The fullstack MCP framework to develop MCP Apps for ChatGPT / Claude & MCP Serve
  ⭐  8,978  awslabs/mcp
           Open source MCP Servers for AWS
  ⭐  4,248  modelcontextprotocol/csharp-sdk
           The official C# SDK for Model Context Protocol servers and clients. Maintained i
  ⭐  2,762  Jpisnice/shadcn-ui-mcp-server
           A mcp server to allow LLMS gain context about shadcn ui component structure,usag
  ⭐  2,356  snyk/agent-scan
           Security scanner for AI agents, MCP servers and agent skills.
  ⭐  2,348  brightdata/brightdata-mcp
           A powerful Model Context Protocol (MCP) server that provides an all-in-one solut

GitHub MCP — get_file_contents()
README.md (first 400 chars):
# Model Context Protocol servers

This repository is a collection of *reference implementations* for the [Model Context Protocol](https://modelcontextprotocol.io/) (MCP), as well as re

In [17]:
# ── Multi-MCP tool router — one function to call them all ────────────────────

def call_tool(tool_name: str, tool_input: dict) -> str:
    """Route tool calls to the appropriate MCP handler."""
    if tool_name == "web_search":
        from ddgs import DDGS
        try:
            with DDGS() as ddgs:
                results = [f"• {r['title']}: {r['body'][:180]}"
                           for r in ddgs.text(tool_input["query"],
                                              max_results=tool_input.get("max_results", 3))]
            return "\n".join(results) or "No results."
        except Exception as e:
            return f"Search error: {e}"

    elif tool_name == "calculator":
        safe_ns = {"__builtins__": {}, "sqrt": math.sqrt, "pi": math.pi,
                   "abs": abs, "round": round, "pow": pow}
        try:
            return str(eval(tool_input["expression"], safe_ns))
        except Exception as e:
            return f"Calc error: {e}"

    elif tool_name == "fetch":
        r = fetch_tool(tool_input["url"], tool_input.get("max_length", 2000))
        return r.get("content", r.get("error", "Fetch failed"))

    elif tool_name == "list_directory":
        return json.dumps(filesystem_tool("list_directory", path=tool_input.get("path", ".")))

    elif tool_name == "write_file":
        path = tool_input.get("path")
        if not path:
            return json.dumps({"error": "write_file requires non-empty 'path'."})
        content = tool_input.get("content")
        if content is None:
            for key in ("text", "body", "contents"):
                if key in tool_input:
                    content = tool_input[key]
                    break
        if content is None:
            return json.dumps({
                "error": "write_file requires 'content' (full file text). "
                         "Call again with both 'path' and 'content'."
            })
        return json.dumps(filesystem_tool("write_file", path=path, content=content))

    elif tool_name == "github_search":
        return json.dumps(github_tool("search_repositories", query=tool_input["query"]))

    elif tool_name == "memory_search":
        hits = memory_retrieve(tool_input["query"], n=tool_input.get("n_results", 3))
        return json.dumps(hits)

    elif tool_name == "sequential_think":
        return think_through(tool_input["problem"])

    return f"Unknown tool: {tool_name}"


# Combined tool registry (our server + external MCPs)
ALL_TOOLS = [
    {"name": "web_search",
     "description": "Search the web for current information.",
     "input_schema": {"type":"object","properties":{"query":{"type":"string"},
                      "max_results":{"type":"integer"}},"required":["query"]}},
    {"name": "calculator",
     "description": "Evaluate a math expression. Supports sqrt, pi, **, %.",
     "input_schema": {"type":"object","properties":{"expression":{"type":"string"}},
                      "required":["expression"]}},
    {"name": "fetch",
     "description": "Fetch a URL and return the page content as plain text.",
     "input_schema": {"type":"object","properties":{"url":{"type":"string"},
                      "max_length":{"type":"integer"}},"required":["url"]}},
    {"name": "list_directory",
     "description": "List files and folders at a filesystem path.",
     "input_schema": {"type":"object","properties":{"path":{"type":"string"}},
                      "required":["path"]}},
    {"name": "write_file",
     "description": "Write text to a file. Always include both path and content in one call — content must be the entire file body, not omitted.",
     "input_schema": {"type":"object","properties":{"path":{"type":"string"},
                      "content":{"type":"string"}},"required":["path","content"]}},
    {"name": "github_search",
     "description": "Search GitHub for repositories matching a query.",
     "input_schema": {"type":"object","properties":{"query":{"type":"string"}},
                      "required":["query"]}},
    {"name": "memory_search",
     "description": "Search your personal memory for past Q&A answers.",
     "input_schema": {"type":"object","properties":{"query":{"type":"string"},
                      "n_results":{"type":"integer"}},"required":["query"]}},
    {"name": "sequential_think",
     "description": "Reason through a complex problem step-by-step before answering.",
     "input_schema": {"type":"object","properties":{"problem":{"type":"string"}},
                      "required":["problem"]}},
]

print(f"Multi-MCP toolbox ready: {[t['name'] for t in ALL_TOOLS]}")

Multi-MCP toolbox ready: ['web_search', 'calculator', 'fetch', 'list_directory', 'write_file', 'github_search', 'memory_search', 'sequential_think']


In [18]:
# ── Multi-MCP ReAct agent — uses any MCP server in a single run ──────────────

def multi_mcp_agent(question: str, max_iter: int = 6) -> str:
    print(f"\n{'═'*65}")
    print(f"  Multi-MCP Agent")
    print(f"  Question: {question}")
    print(f"{'═'*65}\n")

    system = (
        "You are an expert research agent with these tool servers:\n"
        "  web_search      → search the internet\n"
        "  calculator      → do math\n"
        "  fetch           → read any web page\n"
        "  list_directory  → explore the filesystem\n"
        "  write_file      → save results to disk (path + content = full file body)\n"
        "  github_search   → find GitHub repos\n"
        "  memory_search   → search past answers\n"
        "  sequential_think→ reason step-by-step through hard problems\n\n"
        "Strategy: check memory first, then search external sources. "
        "Use sequential_think for complex decisions."
    )

    messages = [{"role": "user", "content": question}]

    for iteration in range(1, max_iter + 1):
        resp = client.messages.create(
            model=MODEL, max_tokens=1000,
            system=system, tools=ALL_TOOLS, messages=messages
        )

        if resp.stop_reason == "end_turn":
            final = next((b.text for b in resp.content if hasattr(b, "text")), "No answer.")
            print(f"\n{'─'*50}\nFINAL ANSWER:\n{final}")
            return final

        messages.append({"role": "assistant", "content": resp.content})
        tool_results = []

        for block in resp.content:
            if block.type == "tool_use":
                print(f"[{iteration}] Calling {block.name}({json.dumps(block.input)[:70]}...)")
                result = call_tool(block.name, block.input)
                print(f"       → {str(result)[:110]}...")
                tool_results.append({"type": "tool_result",
                                     "tool_use_id": block.id,
                                     "content": str(result)})
            elif hasattr(block, "text") and block.text:
                print(f"[{iteration}] Thinking: {block.text[:100]}...")

        if tool_results:
            messages.append({"role": "user", "content": tool_results})

    return "Max iterations reached."


# Try a question that benefits from multiple tool servers
result = multi_mcp_agent(
    "Find the top MCP server projects on GitHub and save a summary report to /tmp/mcp_report.txt"
)


═════════════════════════════════════════════════════════════════
  Multi-MCP Agent
  Question: Find the top MCP server projects on GitHub and save a summary report to /tmp/mcp_report.txt
═════════════════════════════════════════════════════════════════

[1] Thinking: I'll search for the top MCP server projects on GitHub and create a summary report for you....
[1] Calling github_search({"query": "MCP server"}...)
       → {"repos": [{"name": "punkpeye/awesome-mcp-servers", "stars": 86371, "description": "A collection of MCP server...
[2] Thinking: Great! I found the top MCP server projects. Now let me get more detailed information about each of t...
[2] Calling fetch({"url": "https://github.com/punkpeye/awesome-mcp-servers", "max_length...)
       → :root { --tab-size-preference: 4; } pre, code { tab-size: var(--tab-size-preference); } {"locale":"en","featur...
[2] Calling fetch({"url": "https://github.com/microsoft/playwright-mcp", "max_length": 2...)
       → :root { --tab-size-pref

---
# Part 6 — A2A: Agents Talking to Agents

## MCP vs A2A

| | MCP | A2A |
|--|-----|-----|
| **What's exposed** | Tools (functions) | Full agents (entire pipelines) |
| **Who calls it** | Your code | Another agent |
| **Protocol** | JSON-RPC over stdio/HTTP | REST HTTP + polling |
| **Discovery** | `tools/list` | `GET /.well-known/agent.json` |
| **Use case** | "Use this search tool" | "Delegate this whole research task" |

## The A2A handshake

```
Your Agent                              Remote Specialist Agent
    │                                             │
    ├── GET /.well-known/agent.json ─────────────►│  "What can you do?"
    │◄── {name, description, capabilities} ───────┤
    │                                             │
    ├── POST /a2a/tasks ──────────────────────────►│  "Research quantum computing"
    │◄── {task_id: "abc-123"} ────────────────────┤  "I'll get back to you"
    │                                             │
    ├── GET /a2a/tasks/abc-123 ───────────────────►│  (polling)
    │◄── {status: "running"} ─────────────────────┤
    │                                             │
    ├── GET /a2a/tasks/abc-123 ───────────────────►│
    │◄── {status: "complete", result: "..."} ──────┤  Done!
```

## Why it matters

Today your agent calls **tools**.  
Tomorrow your agent delegates **entire research tasks** to specialist agents
running on different servers, in different companies, different countries.

The web of AI agents is just beginning.

In [19]:
# ── A2A Demo: Two agents communicating via the A2A protocol ──────────────────

class SpecialistAgent:
    """Simulates a remote specialist agent (science domain)."""

    AGENT_CARD = {
        "name": "Science Specialist v1",
        "description": "Expert in physics, chemistry, and biology. Delegate research tasks here.",
        "version": "1.0.0",
        "capabilities": ["research", "explain", "calculate"],
        "endpoint": "http://localhost:8001/a2a/tasks",
        "input_schema": {
            "type": "object",
            "properties": {
                "topic":    {"type": "string"},
                "depth":    {"type": "string", "enum": ["brief", "detailed"]},
                "audience": {"type": "string", "enum": ["student", "expert"]}
            },
            "required": ["topic"]
        }
    }

    def __init__(self):
        self.tasks = {}

    def create_task(self, input_data: dict) -> str:
        task_id = str(uuid.uuid4())[:8]
        self.tasks[task_id] = {"status": "pending", "input": input_data, "result": None}
        def _run():
            time.sleep(0.8)
            self.tasks[task_id]["status"] = "running"
            resp = client.messages.create(
                model=MODEL, max_tokens=350,
                system=f"You are a science specialist. Explain clearly for a {input_data.get('audience','student')}. Be {input_data.get('depth','brief')}.",
                messages=[{"role": "user", "content": f"Explain: {input_data.get('topic','')}"}]
            )
            self.tasks[task_id]["result"] = resp.content[0].text
            self.tasks[task_id]["status"] = "complete"
        threading.Thread(target=_run, daemon=True).start()
        return task_id

    def get_task(self, task_id: str) -> dict:
        return self.tasks.get(task_id, {"status": "not_found"})


class OrchestratorAgent:
    """Your local agent — decides when to delegate via A2A."""

    def __init__(self, specialist: SpecialistAgent):
        self.specialist = specialist

    def run(self, question: str) -> str:
        print(f"Orchestrator received: '{question}'")
        card = self.specialist.AGENT_CARD
        print(f"\nDiscovered specialist: {card['name']}")
        print(f"Capabilities: {card['capabilities']}")

        classify = client.messages.create(
            model=MODEL, max_tokens=80,
            system="Reply DELEGATE if this needs deep science expertise, or ANSWER if straightforward.",
            messages=[{"role": "user", "content": question}]
        )
        decision = classify.content[0].text.strip().upper()

        if "DELEGATE" in decision:
            print(f"\nDecision: DELEGATE to {card['name']}")
            task_id = self.specialist.create_task(
                {"topic": question, "depth": "detailed", "audience": "student"})
            print(f"Task created: {task_id}")

            for attempt in range(10):
                time.sleep(0.4)
                task = self.specialist.get_task(task_id)
                print(f"  Poll {attempt+1}: status={task['status']}")
                if task["status"] == "complete":
                    print("Result received from specialist!")
                    return task["result"]
            return "Specialist timed out."
        else:
            print(f"\nDecision: ANSWER directly (no delegation needed)")
            resp = client.messages.create(model=MODEL, max_tokens=250,
                                          messages=[{"role": "user", "content": question}])
            return resp.content[0].text


specialist   = SpecialistAgent()
orchestrator = OrchestratorAgent(specialist)

print("=" * 60)
result = orchestrator.run("Explain how nuclear fusion works and why it's promising for clean energy.")
print("\n" + "=" * 60)
print("Final answer:")
print(result)

Orchestrator received: 'Explain how nuclear fusion works and why it's promising for clean energy.'

Discovered specialist: Science Specialist v1
Capabilities: ['research', 'explain', 'calculate']

Decision: ANSWER directly (no delegation needed)

Final answer:
# How Nuclear Fusion Works

## The Basic Process

Fusion combines two light atomic nuclei (usually hydrogen isotopes) into a heavier nucleus, releasing enormous energy in the process. It's the opposite of fission and powers the sun.

**Key steps:**
- Nuclei must overcome their mutual electrical repulsion by reaching extreme temperatures (millions of degrees)
- At this heat, nuclei collide with enough force to merge
- The combined nucleus has less mass than its parts—this "missing" mass converts to energy via E=mc²

## Why It's Promising

**Clean advantages:**
- Produces no greenhouse gases during operation
- Leaves no long-lived radioactive waste (unlike fission)
- Fuel (hydrogen isotopes) is abundant in seawater
- Inherent safet

In [20]:
# ── A2A over HTTP — what Day 5 will look like ────────────────────────────────
print("A2A HTTP Server (Day 5 — FastAPI):\n")

code_preview = '''
# specialist_server.py  (tomorrow's lab — FastAPI + A2A)
from fastapi import FastAPI, BackgroundTasks
from pydantic import BaseModel
import uuid, asyncio, anthropic

app    = FastAPI()
client = anthropic.Anthropic()
tasks  = {}   # task_id → result

@app.get("/.well-known/agent.json")
async def agent_card():
    """A2A discovery — any agent on the internet finds your capabilities here."""
    return {
        "name": "Science Specialist",
        "description": "Physics, chemistry, biology expert",
        "version": "1.0.0",
        "capabilities": ["research", "explain"],
        "endpoint": "https://your-server.com/a2a/tasks"
    }

@app.post("/a2a/tasks")
async def create_task(body: dict, bg: BackgroundTasks):
    task_id = str(uuid.uuid4())
    tasks[task_id] = {"status": "pending"}
    bg.add_task(run_agent, task_id, body.get("topic", ""))
    return {"task_id": task_id, "status": "accepted"}

@app.get("/a2a/tasks/{task_id}")
async def get_task(task_id: str):
    return tasks.get(task_id, {"error": "not found"})

async def run_agent(task_id: str, topic: str):
    tasks[task_id]["status"] = "running"
    resp = client.messages.create(model="claude-haiku-4-5", max_tokens=400,
                                  messages=[{"role":"user","content":f"Explain: {topic}"}])
    tasks[task_id] = {"status": "complete", "result": resp.content[0].text}
'''

print(code_preview)
print("─" * 55)
print("Launch: uvicorn specialist_server:app --host 0.0.0.0 --port 8001")
print()
print("Any agent can then call:")
print("  POST https://your-server.com/a2a/tasks")
print("  GET  https://your-server.com/.well-known/agent.json")
print()
print("Student A's agent can delegate research to Student B's specialist!")
print("That's a live, distributed, multi-student agent network.")

A2A HTTP Server (Day 5 — FastAPI):


# specialist_server.py  (tomorrow's lab — FastAPI + A2A)
from fastapi import FastAPI, BackgroundTasks
from pydantic import BaseModel
import uuid, asyncio, anthropic

app    = FastAPI()
client = anthropic.Anthropic()
tasks  = {}   # task_id → result

@app.get("/.well-known/agent.json")
async def agent_card():
    """A2A discovery — any agent on the internet finds your capabilities here."""
    return {
        "name": "Science Specialist",
        "description": "Physics, chemistry, biology expert",
        "version": "1.0.0",
        "capabilities": ["research", "explain"],
        "endpoint": "https://your-server.com/a2a/tasks"
    }

@app.post("/a2a/tasks")
async def create_task(body: dict, bg: BackgroundTasks):
    task_id = str(uuid.uuid4())
    tasks[task_id] = {"status": "pending"}
    bg.add_task(run_agent, task_id, body.get("topic", ""))
    return {"task_id": task_id, "status": "accepted"}

@app.get("/a2a/tasks/{task_id}")
async def get_tas

---
# Lab Challenge — Build Your Own MCP-Powered Agent

## Choose your level

### Level 1 — Starter (15 min)
Add a new tool to `lab/mcp_server.py`:

**`summarize_url(url, max_sentences=3)`**
- Calls `fetch_tool` to get the page
- Calls Claude to generate a summary
- Returns: `{url, summary, original_length, sentences}`

Test with: `https://en.wikipedia.org/wiki/Artificial_intelligence`

---

### Level 2 — Intermediate (25 min)
Build a **two-MCP research agent** in LangGraph that:
1. Uses **GitHub MCP** to find top repos on a topic
2. Uses **Fetch MCP** to read the README of the best result
3. Uses **Memory** to store the summary for future questions
4. Returns a structured report

---

### Level 3 — Advanced (40 min)
Build a **personal research assistant** with:
- Persistent memory across runs (`chromadb.PersistentClient('./data/chroma')`)
- `/memory <query>` command to view past answers
- An A2A HTTP endpoint so another student's agent can delegate to yours
- At least two external MCP servers integrated

---

### Bonus — The Wildcard
Pick any server from the ecosystem table and integrate it:
- **Puppeteer MCP** → agent that takes screenshots of websites
- **SQLite MCP** → answer questions about a local database in plain English
- **Slack MCP** → agent that posts research summaries to a Slack channel
- **Google Maps MCP** → agent that recommends places based on your research

In [21]:
# ── Level 1 Starter Template — summarize_url tool ────────────────────────────

def summarize_url_tool(url: str, max_sentences: int = 3) -> dict:
    """
    New MCP tool: fetch a URL and return an AI-generated summary.
    To add to lab/mcp_server.py, decorate with @mcp.tool()
    """
    # Step 1: Fetch the page
    fetch_result = fetch_tool(url, max_length=3000)
    if not fetch_result.get("success"):
        return {"error": fetch_result.get("error", "Fetch failed")}

    content = fetch_result["content"]

    # Step 2: Summarise with Claude
    resp = client.messages.create(
        model=MODEL, max_tokens=300,
        system=f"You are a concise summarizer. Summarize the text in exactly {max_sentences} sentences. Be informative.",
        messages=[{"role": "user", "content": f"Summarize:\n\n{content[:2000]}"}]
    )

    return {
        "url": url,
        "summary": resp.content[0].text,
        "original_length": len(content),
        "sentences": max_sentences
    }


# Test it!
print("Testing summarize_url tool...\n")
result = summarize_url_tool(
    "https://en.wikipedia.org/wiki/Artificial_intelligence",
    max_sentences=3
)
if "summary" in result:
    print(f"URL: {result['url']}")
    print(f"Original: {result['original_length']:,} characters")
    print(f"\nSummary ({result['sentences']} sentences):")
    print(result["summary"])
else:
    print(f"Error: {result}")

Testing summarize_url tool...

URL: https://en.wikipedia.org/wiki/Artificial_intelligence
Original: 3,000 characters

Summary (3 sentences):
# Artificial Intelligence Summary

Artificial intelligence refers to computer systems designed to perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making. AI encompasses various approaches including machine learning, neural networks, and deep learning, enabling machines to improve their performance through experience and data analysis. Modern AI applications span numerous fields including healthcare, finance, autonomous vehicles, and natural language processing, making it one of the most transformative technologies of the 21st century.


In [22]:
# ── Level 2 Template — Two-MCP LangGraph Research Agent ─────────────────────

class ResearchState(TypedDict):
    topic:          str
    github_repos:   list
    readme_content: str
    summary:        str
    stored:         bool
    log:            list


def rnode_github(state):
    result = github_tool("search_repositories", query=f"{state['topic']} stars:>30")
    repos = result.get("repos", [])
    return {**state, "github_repos": repos,
            "log": state.get("log",[]) + [f"[GitHub] {len(repos)} repos found"]}

def rnode_fetch_readme(state):
    if not state["github_repos"]:
        return {**state, "readme_content": "No repos found.",
                "log": state.get("log",[]) + ["[Fetch] no repos to fetch"]}
    top = state["github_repos"][0]
    try:
        owner, repo = top["name"].split("/")
    except ValueError:
        return {**state, "readme_content": "Invalid repo name.",
                "log": state.get("log",[]) + ["[Fetch] invalid repo name"]}
    result = github_tool("get_file_contents", owner=owner, repo=repo, path="README.md")
    content = result.get("content", "README not available.")
    return {**state, "readme_content": content,
            "log": state.get("log",[]) + [f"[Fetch] README from {top['name']}"]}

def rnode_summarize(state):
    repos_text = "\n".join(
        f"- {r['name']} (⭐{r['stars']:,}): {r['description']}"
        for r in state["github_repos"][:3]
    )
    resp = client.messages.create(
        model=MODEL, max_tokens=400,
        system="You are a technical researcher. Write a clear summary of the topic based on top GitHub projects.",
        messages=[{"role": "user", "content":
                   f"Topic: {state['topic']}\n\nTop repos:\n{repos_text}\n\nREADME excerpt:\n{state['readme_content'][:1000]}"}]
    )
    return {**state, "summary": resp.content[0].text,
            "log": state.get("log",[]) + ["[Summary] written"]}

def rnode_store(state):
    msg = memory_store(state["topic"], state["summary"], score=8)
    return {**state, "stored": True, "log": state.get("log",[]) + [f"[Memory] {msg}"]}


rg = StateGraph(ResearchState)
for name, fn in [("github", rnode_github), ("readme", rnode_fetch_readme),
                 ("summarize", rnode_summarize), ("store", rnode_store)]:
    rg.add_node(name, fn)
rg.add_edge(START, "github")
rg.add_edge("github", "readme")
rg.add_edge("readme", "summarize")
rg.add_edge("summarize", "store")
rg.add_edge("store", END)

research_agent = rg.compile()

print("Running two-MCP research agent...\n")
result = research_agent.invoke({
    "topic": "model context protocol MCP",
    "github_repos": [], "readme_content": "", "summary": "", "stored": False, "log": []
})
for entry in result["log"]:
    print(f"  {entry}")
print(f"\nSummary:\n{result['summary']}")

Running two-MCP research agent...

  [GitHub] 6 repos found
  [Fetch] README from microsoft/mcp-for-beginners
  [Summary] written
  [Memory] Stored (id=2a3964df). Total: 4

Summary:
# Model Context Protocol (MCP) - Technical Summary

## Overview

**Model Context Protocol (MCP)** is an open standard that enables AI models and applications to securely interact with external tools, data sources, and services. It provides a standardized way for large language models (LLMs) to access contextual information and execute tasks beyond their native capabilities.

## Key Characteristics

### Core Purpose
MCP acts as a bridge between AI models and external systems, allowing:
- **Tool Integration**: Expose custom functions and APIs as callable tools
- **Data Access**: Retrieve contextual information from various sources
- **Service Connectivity**: Connect LLMs to external applications and databases

### Authentication & Security
The protocol includes built-in support for authentication, ensuring se

---
# Day 4 Complete — What You Built

## Component summary

| Component | What it does |
|-----------|-------------|
| **ChromaDB** | Vector database — persistent episodic memory |
| **RAG pipeline** | Retrieve past answers → better generation |
| **Custom MCP server** | Exposes 5 tools any agent can call |
| **Fetch MCP** | Browse any web page from inside your agent |
| **Filesystem MCP** | Read/write files — agents that leave artifacts |
| **Sequential Thinking MCP** | Auditable, step-by-step reasoning chains |
| **GitHub MCP** | Search repos, read files, manage issues |
| **Multi-MCP ReAct agent** | Routes across 8 tool servers in one run |
| **A2A protocol** | Delegate entire tasks to specialist agents |

## What's next — Day 5

```
Day 5 architecture:

┌──────────────┐    webhook    ┌─────────────────────────────┐
│   WhatsApp   │ ────────────► │   FastAPI + Uvicorn         │
│   Business   │ ◄──────────── │                             │
└──────────────┘               │  ┌──────────────────────┐   │
                               │  │   Day 4 RAG Agent    │   │
                               │  │   + MCP server       │   │
                               │  │   + ChromaDB         │   │
                               │  └──────────────────────┘   │
                               └─────────────────────────────┘
```

Your RAG-powered, MCP-connected, multi-agent system will be accessible
from **any phone in the world** via WhatsApp.

## Key takeaways

1. **Memory = power** — agents that remember grow smarter with every interaction
2. **MCP = reuse** — build tools once, any agent anywhere can call them
3. **External MCPs** — GitHub, web, filesystem are already built; just connect
4. **A2A** — the future is a web of collaborating specialist agents
5. **Sequential Thinking** — auditable reasoning chains make complex decisions transparent

See you tomorrow for deployment day! 🚀